# Phase 3: Feature Engineering and Baseline Forecasting

Reuses the validated workbook loaders and availability selector. No advanced models are trained. All generated data, forecasts and figures stay under ignored data/phase3/. The September 30, 2026 decision uses assumption-based availability, not verified historical vintages. FX has a provisional conservative lag scenario, documented separately from source-supported lags.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, display

root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src/schneider_electric_forecasting").exists()
)
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "scripts"))

from commands.run_phase3 import main  # noqa: E402

results = main()

## Origin-specific feature and supervised datasets

Lags use the last observable month as their anchor; latest value and observation age are explicit. Rolling windows include only observations selected at that origin. Missing periods remain null. Labels and estimated label publication times are separate from model inputs. Training labels must pass training_rows at each fold cutoff; no global imputation, scaling or feature selection is fitted.

In [ ]:
display(
    results["dataset"]
    .drop(columns=["target_value", "target_available_at_estimated"])
    .tail()
)
print("Supervised rows:", len(results["dataset"]))
display(
    results["dataset"][
        [
            "series_id",
            "origin",
            "horizon",
            "target_month",
            "target_value",
            "target_available_at_estimated",
        ]
    ].tail(36)
)

## Paired rolling-origin baseline evaluation

LOCF is the mandatory primary benchmark. Seasonal naive repeats the latest observable target-month season; drift uses slope per elapsed calendar month; historical mean return compounds the expanding mean consecutive monthly return. All forecast the actual target month, including publication delay. Each series/horizon uses identical eligible evaluation keys for all four models. MASE uses origin-specific training-only one-step naive MAE (m=1). Bias means forecast minus actual. Actual labels unavailable by the final decision are excluded, and final forecasts are never scored against invented actuals.

In [ ]:
display(results["scores"].sort_values(["series_id", "horizon", "MAE"]))
print("Paired-window counts by material and horizon")
display(
    results["scores"].query("model == 'LOCF'")[
        ["series_id", "horizon", "valid_windows", "first_origin", "last_origin"]
    ]
)

## Forecasts versus observed outcomes

Plots use each material's original units. Evaluation errors overlap in time, so descriptive model rankings do not establish statistical superiority.

In [ ]:
for path in sorted(results["output"].glob("forecast_actual_*.png")):
    display(Image(filename=str(path)))

## Final September 2026 forecasts

Targets are December 2026, March 2027 and June 2027. Last available RM month is June; effective distances are therefore six, nine and twelve months. These are point forecasts with no fabricated predictive intervals. Units are not pooled across series.

In [ ]:
display(results["forecasts"])

## Material-specific recommendations and readiness

Retain LOCF regardless of which baseline wins historical MAE. Recheck apparent improvements with fresh origins and nested time-series validation before model selection. PPI proxies and cost drivers remain distinct from grade prices. Missing BOM mappings, mass inconsistencies and monetary weights are not filled.

In [ ]:
display(results["recommendations"])
print((results["output"] / "phase3_report.md").read_text())